In [ ]:
conda install biopython


In [ ]:
print("Biopython has been installed successfully.")

In [1]:
from Bio.PDB import PDBParser
import pandas as pd

charge_map = {
    "LYS": +1,
    "ARG": +1,
    "ASP": -1,
    "GLU": -1,
    "HIS": 0
}

protein_residues = {
    "ALA","ARG","ASN","ASP","CYS",
    "GLN","GLU","GLY","HIS","ILE",
    "LEU","LYS","MET","PHE","PRO",
    "SER","THR","TRP","TYR","VAL"
}

pdb_file = "2CV5.pdb"

parser = PDBParser(QUIET=True)
structure = parser.get_structure("NCP", pdb_file)

records = []

chain_charge = {}

for model in structure:
    for chain in model:

        chain_id = chain.id
        total_charge = 0

        for residue in chain:

            if residue.id[0] != " ":
                continue

            resname = residue.resname

            if resname not in protein_residues:
                continue

            resid = residue.id[1]

            charge = charge_map.get(resname, 0)

            total_charge += charge

            records.append({
                "Chain": chain_id,
                "Residue_Number": resid,
                "Residue_Name": resname,
                "Charge": charge
            })

        chain_charge[chain_id] = total_charge

# =========================
# Save residue table
# =========================
df = pd.DataFrame(records)
df.to_csv("charged_residues_2CV5.csv", index=False)

# =========================
# Print total charges
# =========================
print("\n=== Total Charge Per Chain for 2CV5 ===")

grand_total = 0

for chain, charge in chain_charge.items():
    print(f"Chain {chain}: {charge}")

    grand_total += charge

print("\nTotal Histone Charge:", grand_total)


=== Total Charge Per Chain for 2CV5 ===
Chain I: 0
Chain J: 0
Chain A: 8
Chain B: 10
Chain C: 11
Chain D: 9
Chain E: 9
Chain F: 12
Chain G: 9
Chain H: 7

Total Histone Charge: 75


In [2]:
from Bio.PDB import PDBParser
import pandas as pd

charge_map = {
    "LYS": +1,
    "ARG": +1,
    "ASP": -1,
    "GLU": -1,
    "HIS": 0
}

protein_residues = {
    "ALA","ARG","ASN","ASP","CYS",
    "GLN","GLU","GLY","HIS","ILE",
    "LEU","LYS","MET","PHE","PRO",
    "SER","THR","TRP","TYR","VAL"
}

pdb_file = "1KX5.pdb"

parser = PDBParser(QUIET=True)
structure = parser.get_structure("NCP", pdb_file)

records = []

chain_charge = {}

for model in structure:
    for chain in model:

        chain_id = chain.id
        total_charge = 0

        for residue in chain:

            if residue.id[0] != " ":
                continue

            resname = residue.resname

            if resname not in protein_residues:
                continue

            resid = residue.id[1]

            charge = charge_map.get(resname, 0)

            total_charge += charge

            records.append({
                "Chain": chain_id,
                "Residue_Number": resid,
                "Residue_Name": resname,
                "Charge": charge
            })

        chain_charge[chain_id] = total_charge

# =========================
# Save residue table
# =========================
df = pd.DataFrame(records)
df.to_csv("charged_residues_1KX5.csv", index=False)

# =========================
# Print total charges
# =========================
print("\n=== Total Charge Per Chain for 1KX5 ===")

grand_total = 0

for chain, charge in chain_charge.items():
    print(f"Chain {chain}: {charge}")

    grand_total += charge

print("\nTotal Histone Charge:", grand_total)


=== Total Charge Per Chain for 1KX5 ===
Chain I: 0
Chain J: 0
Chain A: 20
Chain B: 18
Chain C: 17
Chain D: 19
Chain E: 20
Chain F: 18
Chain G: 17
Chain H: 19

Total Histone Charge: 148


In [3]:
from Bio import SeqIO
import pandas as pd
import re

# =========================================================
# Residues charged at physiological pH
# =========================================================

positive_residues = ['K', 'R']   # Lysine, Arginine
negative_residues = ['D', 'E']   # Aspartate, Glutamate

# Histidine optional
count_histidine = False

# =========================================================
# FASTA FILES
# =========================================================

fasta_files = [
    "rcsb_pdb_1KX5.fasta",
    "rcsb_pdb_2CV5.fasta"
]

# =========================================================
# PROCESS EACH FASTA
# =========================================================

for fasta_file in fasta_files:

    print("\n" + "="*70)
    print(f"PROCESSING : {fasta_file}")
    print("="*70)

    results = []
    grand_total = 0

    for record in SeqIO.parse(fasta_file, "fasta"):

        sequence = str(record.seq)

        # =================================================
        # Parse FASTA header
        # =================================================

        description = record.description
        parts = description.split("|")

        entry = parts[0]

        chains_info = parts[1] if len(parts) > 1 else "Unknown"
        protein = parts[2] if len(parts) > 2 else "Unknown"

        # =================================================
        # Skip DNA chains
        # =================================================

        # Skip DNA chains only
        if "DNA" in description.upper():
            continue

        # =================================================
        # Count copies/chains
        # Example:
        # Chains C[auth A], G[auth E]
        # =================================================

        chain_matches = re.findall(r'\b[A-Z]\[', chains_info)

        n_chains = len(chain_matches)

        # Safety fallback
        if n_chains == 0:
            n_chains = 1

        # =================================================
        # Count charges for ONE chain
        # =================================================

        pos_count = sum(sequence.count(x) for x in positive_residues)

        neg_count = sum(sequence.count(x) for x in negative_residues)

        # Optional Histidine
        if count_histidine:
            pos_count += sequence.count('H')

        single_chain_charge = pos_count - neg_count

        # =================================================
        # Total charge including copies
        # =================================================

        total_charge = single_chain_charge * n_chains

        grand_total += total_charge

        # =================================================
        # Store results
        # =================================================

        results.append({
            "Protein"             : protein,
            "Chains"              : chains_info,
            "Copies"              : n_chains,
            "Sequence_Length"     : len(sequence),
            "Positive_Residues"   : pos_count,
            "Negative_Residues"   : neg_count,
            "Single_Chain_Charge" : single_chain_charge,
            "Total_Charge"        : total_charge
        })

    # =====================================================
    # Create dataframe
    # =====================================================

    df = pd.DataFrame(results)

    # =====================================================
    # Print results nicely
    # =====================================================

    print("\nHistone Charge Summary\n")

    print(df.to_string(index=False))

    print("\n" + "-"*70)
    print(f"TOTAL HISTONE CHARGE FOR {fasta_file} = {grand_total}")
    print("-"*70)

    # =====================================================
    # Save CSV
    # =====================================================

    csv_name = fasta_file.replace(".fasta", "_charge_summary.csv")

    df.to_csv(csv_name, index=False)

    print(f"\nSaved CSV : {csv_name}")


PROCESSING : rcsb_pdb_1KX5.fasta

Histone Charge Summary

      Protein                      Chains  Copies  Sequence_Length  Positive_Residues  Negative_Residues  Single_Chain_Charge  Total_Charge
   histone H3 Chains C[auth A], G[auth E]       2              135                 31                 11                   20            40
   histone H4 Chains D[auth B], H[auth F]       2              102                 25                  7                   18            36
histone H2A.1 Chains E[auth C], I[auth G]       2              128                 26                  9                   17            34
histone H2B.2 Chains F[auth D], J[auth H]       2              125                 28                 10                   18            36

----------------------------------------------------------------------
TOTAL HISTONE CHARGE FOR rcsb_pdb_1KX5.fasta = 146
----------------------------------------------------------------------

Saved CSV : rcsb_pdb_1KX5_charge_summary.csv



In [4]:
from Bio.PDB import PDBParser

pdb_file = "2CV5.pdb"

parser = PDBParser(QUIET=True)
structure = parser.get_structure("NCP", pdb_file)

for chain in structure[0]:

    residues = [
        res.id[1]
        for res in chain
        if res.id[0] == " "
    ]

    if len(residues) == 0:
        continue

    print(f"\nChain {chain.id}")

    print("First residue :", min(residues))
    print("Last residue  :", max(residues))
    print("Total resolved:", len(residues))


Chain I
First residue : 1
Last residue  : 146
Total resolved: 146

Chain J
First residue : 147
Last residue  : 292
Total resolved: 146

Chain A
First residue : 38
Last residue  : 134
Total resolved: 97

Chain B
First residue : 25
Last residue  : 102
Total resolved: 78

Chain C
First residue : 11
Last residue  : 118
Total resolved: 108

Chain D
First residue : 27
Last residue  : 122
Total resolved: 96

Chain E
First residue : 37
Last residue  : 135
Total resolved: 99

Chain F
First residue : 18
Last residue  : 102
Total resolved: 85

Chain G
First residue : 15
Last residue  : 118
Total resolved: 104

Chain H
First residue : 28
Last residue  : 121
Total resolved: 94


In [6]:
from Bio import SeqIO
import pandas as pd
import re
from collections import Counter

# ============================================================
# CHARGE DEFINITIONS AT PHYSIOLOGICAL pH
# ============================================================

charge_map = {
    'K': +1,   # Lysine
    'R': +1,   # Arginine
    'D': -1,   # Aspartate
    'E': -1    # Glutamate
}

# Optional Histidine handling
include_histidine = False

if include_histidine:
    charge_map['H'] = +1

# ============================================================
# FASTA FILES
# ============================================================

fasta_files = [
    "rcsb_pdb_1KX5.fasta",
    "rcsb_pdb_2CV5.fasta"
]

# ============================================================
# PROCESS EACH FASTA FILE
# ============================================================

for fasta_file in fasta_files:

    print("\n" + "="*80)
    print(f"PROCESSING FILE : {fasta_file}")
    print("="*80)

    # --------------------------------------------------------
    # Store outputs
    # --------------------------------------------------------

    residue_records = []
    summary_records = []

    grand_total_charge = 0

    # ========================================================
    # READ FASTA
    # ========================================================

    for record in SeqIO.parse(fasta_file, "fasta"):

        sequence = str(record.seq)

        description = record.description

        parts = description.split("|")

        entry = parts[0]

        chains_info = parts[1] if len(parts) > 1 else "Unknown"

        protein_name = parts[2] if len(parts) > 2 else "Unknown"

        # ====================================================
        # SKIP DNA CHAINS
        # ====================================================

        if "DNA" in description.upper():
            continue

        # ====================================================
        # COUNT NUMBER OF IDENTICAL CHAINS
        # Example:
        # Chains C[auth A], G[auth E]
        # ====================================================

        chain_matches = re.findall(r'\b[A-Z]\[', chains_info)

        n_chains = len(chain_matches)

        if n_chains == 0:
            n_chains = 1

        # ====================================================
        # CHARGE CALCULATION
        # ====================================================

        positive_count = 0
        negative_count = 0
        net_charge_single = 0

        # Residue-wise processing
        for i, residue in enumerate(sequence, start=1):

            charge = charge_map.get(residue, 0)

            # Positive / negative counts
            if charge > 0:
                positive_count += 1

            elif charge < 0:
                negative_count += 1

            net_charge_single += charge

            # ------------------------------------------------
            # Store residue-wise information
            # ------------------------------------------------

            residue_records.append({
                "FASTA_File"       : fasta_file,
                "Protein"          : protein_name,
                "Chains"           : chains_info,
                "Copies"           : n_chains,
                "Residue_Number"   : i,
                "Residue"          : residue,
                "Charge"           : charge
            })

        # ====================================================
        # TOTAL CHARGE INCLUDING MULTIPLE COPIES
        # ====================================================

        total_charge = net_charge_single * n_chains

        grand_total_charge += total_charge

        # ====================================================
        # STORE SUMMARY INFORMATION
        # ====================================================

        summary_records.append({
            "Protein"              : protein_name,
            "Chains"               : chains_info,
            "Copies"               : n_chains,
            "Sequence_Length"      : len(sequence),
            "Positive_Residues"    : positive_count,
            "Negative_Residues"    : negative_count,
            "Single_Chain_Charge"  : net_charge_single,
            "Total_Charge"         : total_charge
        })

    # ========================================================
    # CREATE DATAFRAMES
    # ========================================================

    residue_df = pd.DataFrame(residue_records)

    summary_df = pd.DataFrame(summary_records)

    # ========================================================
    # PRINT SUMMARY
    # ========================================================

    print("\nHISTONE CHARGE SUMMARY\n")

    print(summary_df.to_string(index=False))

    print("\n" + "-"*80)
    print(f"TOTAL HISTONE OCTAMER CHARGE = {grand_total_charge}")
    print("-"*80)

    # ========================================================
    # SAVE OUTPUT FILES
    # ========================================================

    base_name = fasta_file.replace(".fasta", "")

    residue_csv = f"{base_name}_residue_charges.csv"

    summary_csv = f"{base_name}_charge_summary.csv"

    residue_df.to_csv(residue_csv, index=False)

    summary_df.to_csv(summary_csv, index=False)

    print(f"\nSaved residue-wise charges : {residue_csv}")

    print(f"Saved summary table        : {summary_csv}")



PROCESSING FILE : rcsb_pdb_1KX5.fasta

HISTONE CHARGE SUMMARY

      Protein                      Chains  Copies  Sequence_Length  Positive_Residues  Negative_Residues  Single_Chain_Charge  Total_Charge
   histone H3 Chains C[auth A], G[auth E]       2              135                 31                 11                   20            40
   histone H4 Chains D[auth B], H[auth F]       2              102                 25                  7                   18            36
histone H2A.1 Chains E[auth C], I[auth G]       2              128                 26                  9                   17            34
histone H2B.2 Chains F[auth D], J[auth H]       2              125                 28                 10                   18            36

--------------------------------------------------------------------------------
TOTAL HISTONE OCTAMER CHARGE = 146
--------------------------------------------------------------------------------

Saved residue-wise charges : rcsb_pdb